# **ACE (Automatic Concept-based Explanations): Практика**

**Всем привет.**

В уроке «TCAV: Практика» концепт «полосы» был задан заранее: кто-то собрал папку `striped` с полосатыми картинками, и TCAV проверял, важен ли этот концепт для класса «зебра». ACE (Automatic Concept-based Explanations, [Ghorbani et al., 2019](https://arxiv.org/abs/1902.03129)) убирает этот шаг: концепты-кандидаты метод находит сам, нарезая изображения класса на куски и группируя похожие куски по активациям сети. Имя найденному концепту дает уже человек, глядя на картинки.

Авторы статьи выложили [код на TensorFlow](https://github.com/amiratag/ACE). Здесь — упрощенная реализация тех же трех шагов на PyTorch и `captum`, поверх данных и модели из урока «TCAV: Практика»: зебры из `tcav.zip`, GoogLeNet с весами ImageNet, класс «зебра» (индекс 340), слой `inception4e`.

В этой практике вы:
- нарежете изображения зебр на сегменты алгоритмом SLIC на двух разрешениях и подадите каждый сегмент сети на сером фоне;
- переведете сегменты в активации слоя `inception4e` и сгруппируете их KMeans — каждый кластер станет кандидатом в концепт;
- отбросите кластеры, которые собраны из малого числа изображений, и посмотрите на сегменты оставшихся;
- посчитаете TCAV-score каждого оставшегося кластера против случайных изображений и упорядочите концепты по важности;
- проверите, насколько список концептов зависит от числа кластеров $k$.

**Приятного кодинга!**

Как всегда, прежде установим необходимые зависимости. В Colab уже есть `scikit-image`, `scikit-learn` и `torch`; доставить нужно только `captum`.

In [ ]:
!pip install captum==0.9.0 -q

In [ ]:
import os, glob, random, shutil, warnings

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torchvision
from torchvision import transforms

from skimage.segmentation import slic, mark_boundaries
from sklearn.cluster import KMeans

from captum.concept import TCAV, Concept

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## Данные и модель

Данные те же, что в уроке «TCAV: Практика»: папка `tcav/data/zebra` с зебрами и папки `tcav/concepts/random500_0`, `tcav/concepts/random500_1` со случайными фотографиями с Wikimedia Commons (снимки Unsplash, CC0). Источник и лицензия каждой фотографии — в файле `tcav/CREDITS.md`. Папки `striped`, `zigzagged`, `dotted` нам не понадобятся: концепты мы будем искать сами.

In [ ]:
!wget -q https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/tcav.zip -O tcav.zip

!unzip -q -o 'tcav.zip'

Чтобы тетрадь выполнялась на CPU в Colab несколько минут, возьмем первые 20 зебр. Каждое изображение приводим к размеру 224×224 так же, как в TCAV (`Resize(256)` и `CenterCrop(224)`), но пока без нормализации: нарезать на сегменты будем обычную картинку.

In [ ]:
general_path = os.path.abspath('tcav')  # папка tcav в рабочей папке: на Colab это /content/tcav

crop = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
normalize = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

def load_images(folder, n=None):
    files = sorted(glob.glob(os.path.join(general_path, folder, '*.jpg')))[:n]  # постоянный порядок файлов
    return [np.array(crop(Image.open(f).convert('RGB'))) for f in files]      # uint8, (224, 224, 3)

N_IMAGES = 20
zebra_imgs = load_images('data/zebra', N_IMAGES)
print(f'Зебр: {len(zebra_imgs)}, размер изображения: {zebra_imgs[0].shape}')

In [ ]:
model = torchvision.models.googlenet(weights=torchvision.models.GoogLeNet_Weights.IMAGENET1K_V1).to(device).eval()

zebra_ind = 340
zebra_tensors = torch.stack([normalize(img) for img in zebra_imgs]).to(device)
with torch.no_grad():
    zebra_pred = model(zebra_tensors).argmax(1)
print(f'Модель относит к классу «зебра» {(zebra_pred == zebra_ind).sum().item()} из {N_IMAGES} изображений')

## Шаг 1. Нарезка на сегменты

ACE режет каждое изображение алгоритмом [SLIC](https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.slic): он группирует соседние пиксели близкого цвета в суперпиксели, а параметр `n_segments` задает, сколько примерно кусков получится. В статье берут три разрешения, чтобы концепт мог оказаться и крупной частью объекта, и мелкой текстурой; в коде авторов это `n_segments` = 15, 50 и 80 при `compactness=20`. Мы возьмем два из них, 15 и 50: на 20 зебрах разрешение 80 дает больше половины всех сегментов и примерно удваивает время выполнения тетради.

Посмотрим, как одно изображение режется на этих двух разрешениях.

In [ ]:
RESOLUTIONS = [15, 50]

def segment(img, n_segments):
    return slic(img, n_segments=n_segments, compactness=20, sigma=1, start_label=0)

fig, axs = plt.subplots(1, len(RESOLUTIONS), figsize=(10, 5))
for ax, n_seg in zip(axs, RESOLUTIONS):
    labels = segment(zebra_imgs[7], n_seg)
    ax.imshow(mark_boundaries(zebra_imgs[7], labels))
    ax.set_title(f'n_segments={n_seg}: {labels.max() + 1} segments')
    ax.axis('off')
plt.show()

Дальше каждый сегмент подается сети отдельно. ACE делает это так: пиксели вне сегмента заливает серым цветом со значением 117 по каждому каналу, вырезает прямоугольник, описанный вокруг сегмента (bounding box), и растягивает его до входного размера сети 224×224.

**Задание 1.** Допишите функцию `segment_patch`: на входе изображение `img` (uint8, 224×224×3) и булева маска сегмента `mask` (224×224), на выходе — массив uint8 224×224×3: сегмент на сером фоне, вырезанный по bounding box и растянутый до 224×224 (`Image.fromarray(...).resize((224, 224), Image.BICUBIC)`).

In [ ]:
GRAY = 117

def segment_patch(img, mask):
    # Ваш код здесь
    pass

In [ ]:
EXAMPLE = 7
labels = segment(zebra_imgs[EXAMPLE], 15)
example_masks = [labels == labels[y, x] for y, x in [(60, 100), (130, 200), (40, 30), (215, 130)]]

fig, axs = plt.subplots(2, 4, figsize=(16, 8))
for j, mask in enumerate(example_masks):
    axs[0, j].imshow(mark_boundaries(zebra_imgs[EXAMPLE], mask.astype(int)))
    axs[0, j].set_title(f'segment {j}: {mask.mean():.1%} of the image')
    axs[1, j].imshow(segment_patch(zebra_imgs[EXAMPLE], mask))
    axs[0, j].axis('off'); axs[1, j].axis('off')
plt.show()

**Вопрос для самопроверки.** Почему фон серый со значением 117, а не черный? Посчитайте, во что превращаются 117 и 0 после нормализации `normalize`.

<details><summary>Проверить себя</summary>

После нормализации ImageNet пиксель 117 переходит в значения около нуля по всем трем каналам: −0.11, 0.01 и 0.23 (расчет в коде ниже), то есть почти в «средний» вход, на котором сеть обучалась. Черный пиксель 0 переходит в −2.12, −2.04 и −1.80: сеть увидела бы вокруг сегмента сильный сигнал темной рамки, которого нет в исходном изображении.

</details>

In [ ]:
mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
print('117 ->', np.round((117 / 255 - mean) / std, 2))
print('  0 ->', np.round((0 - mean) / std, 2))

Перед тем как передать сегменты сети, ACE отбрасывает два вида лишних. Первый — совсем мелкие сегменты, меньше 0.1% площади изображения: после растяжения до 224×224 от них остается несколько размытых пикселей. Второй — повторы: крупный кусок на разрешении 15 и кусок на разрешении 50 могут почти совпадать, и тогда один и тот же участок изображения попал бы в данные дважды. В коде ACE сегмент считается повтором, если его коэффициент Жаккара с каким-либо сегментом того же изображения, найденным на предыдущем разрешении, больше 0.5. Коэффициент Жаккара двух масок — число пикселей, попавших в обе маски, деленное на число пикселей, попавших хотя бы в одну.

**Задание 2.** Допишите функцию `is_duplicate`: на входе булева маска `mask` и список масок `seen_masks` того же изображения с предыдущих разрешений, на выходе `True`, если коэффициент Жаккара `mask` хотя бы с одной маской из `seen_masks` больше 0.5. Затем выполните код ниже: он нарежет все 20 зебр на двух разрешениях и оставит сегменты, прошедшие оба фильтра.

In [ ]:
MIN_AREA = 0.001

def is_duplicate(mask, seen_masks):
    # Ваш код здесь
    pass

In [ ]:
patches, owners, scales = [], [], []   # сегмент на сером фоне, номер изображения, разрешение SLIC
n_raw, n_small, n_dup = 0, 0, 0
for i, img in enumerate(zebra_imgs):
    seen_masks = []                      # маски этого изображения с предыдущих разрешений
    for n_seg in RESOLUTIONS:
        labels = segment(img, n_seg)
        new_masks = []
        for s in np.unique(labels):
            mask = labels == s
            n_raw += 1
            if mask.mean() <= MIN_AREA:
                n_small += 1
            elif is_duplicate(mask, seen_masks):
                n_dup += 1
            else:
                new_masks.append(mask)
                patches.append(segment_patch(img, mask))
                owners.append(i)
                scales.append(n_seg)
        seen_masks += new_masks
owners, scales = np.array(owners), np.array(scales)

print(f'Сегментов всего: {n_raw}; мелких: {n_small}; повторов: {n_dup}; осталось: {len(patches)}')
for n_seg in RESOLUTIONS:
    print(f'  n_segments={n_seg}: осталось {(scales == n_seg).sum()} сегментов')

**Вопрос для самопроверки.** Сколько сегментов SLIC дал на всех 20 изображениях, сколько из них отброшено как повторы и сколько осталось? На каком разрешении отброшены повторы и почему не на другом?

<details><summary>Проверить себя</summary>

SLIC дал 982 сегмента: 238 на разрешении 15 и 744 на разрешении 50. Мелких сегментов, меньше 0.1% площади изображения, нет ни одного; 28 сегментов отброшены как повторы, осталось 954. Все 28 повторов отброшены на разрешении 50: сегмент сравнивается только с масками того же изображения с предыдущих разрешений, а на первом разрешении, 15, сравнивать не с чем; сегменты одного разрешения не пересекаются друг с другом.

</details>

## Шаг 2. Активации и кластеры

Каждый сегмент пропускаем через GoogLeNet и забираем активацию слоя `inception4e` хуком. Карта активаций имеет форму 832×14×14; усредняем ее по пространству и получаем вектор из 832 чисел на сегмент. Заодно сохраним, какой класс модель предсказывает для самого сегмента: это пригодится в конце.

In [ ]:
LAYER = 'inception4e'
store = {}
hook = getattr(model, LAYER).register_forward_hook(lambda m, inp, out: store.update(act=out))

feats, seg_pred = [], []
with torch.no_grad():
    for b in range(0, len(patches), 64):
        x = torch.stack([normalize(p) for p in patches[b:b + 64]]).to(device)
        seg_pred.append(model(x).argmax(1))
        feats.append(store['act'].mean(dim=(2, 3)))   # усреднение по пространству 14×14
hook.remove()

feats = torch.cat(feats).cpu().numpy()
seg_pred = torch.cat(seg_pred).cpu().numpy()
feats.shape

Теперь сегменты со всех изображений группируем KMeans. Каждый кластер — кандидат в концепт: набор кусков, которые слой `inception4e` считает похожими. В статье и коде авторов $k=25$; у нас 20 изображений и два разрешения SLIC, сегментов меньше, и мы берем $k=12$.

In [ ]:
K = 12
kmeans = KMeans(n_clusters=K, n_init=10, random_state=0).fit(feats)
cluster_of = kmeans.labels_              # номер кластера каждого сегмента
dist = kmeans.transform(feats)           # расстояние каждого сегмента до центра каждого кластера
np.bincount(cluster_of)

## Шаг 3. Чистка кластеров

Не каждый кластер годится в концепт. Если все сегменты кластера взяты из одного-двух изображений, это скорее особенность конкретных фотографий (например, одинаковый фон), чем то, что сеть видит в классе «зебра» вообще. ACE оставляет кластеры, где сегментов достаточно много и они взяты из многих разных изображений.

**Задание 3.** Допишите функцию `select_clusters`: она возвращает список номеров кластеров, в которых не меньше `min_size` сегментов и эти сегменты взяты не меньше чем из `min_images` разных изображений (номер изображения сегмента лежит в `owners`). Возьмем `min_size=20` и `min_images=10`, то есть половину из 20 зебр. Это упрощенный вариант фильтра ACE: в коде авторов условий три, и они сравнивают число изображений в кластере и с размером кластера, и с числом всех изображений класса.

In [ ]:
MIN_SIZE, MIN_IMAGES = 20, 10

def select_clusters(cluster_of, owners, min_size=MIN_SIZE, min_images=MIN_IMAGES):
    # Ваш код здесь
    pass

In [ ]:
def cluster_table(cluster_of, kept):
    print('cluster | segments | images | n_segments 15/50 | predicted "zebra" | kept')
    for c in range(cluster_of.max() + 1):
        m = cluster_of == c
        per_scale = '/'.join(str((scales[m] == r).sum()) for r in RESOLUTIONS)
        print(f'{c:7d} | {m.sum():8d} | {len(np.unique(owners[m])):6d} | {per_scale:>15} | '
              f'{(seg_pred[m] == zebra_ind).mean():17.0%} | {"yes" if c in kept else "no"}')

kept = select_clusters(cluster_of, owners)
cluster_table(cluster_of, kept)
print(f'\nKept clusters: {len(kept)} of {K}: {kept}')

**Вопрос для самопроверки.** Сколько кластеров из 12 прошло фильтр? Какой из отброшенных кластеров собран из наименьшего числа изображений?

<details><summary>Проверить себя</summary>

Фильтр прошли 10 кластеров из 12. Отброшены кластер 1 (64 сегмента из 8 изображений) и кластер 9 (58 сегментов из 7 изображений); меньше всего изображений у кластера 9: его сегменты взяты всего с семи фотографий, и такой кластер скорее описывает эти фотографии, чем класс «зебра».

</details>

Посмотрим на сами кандидаты в концепты. Для каждого оставшегося кластера покажем 6 сегментов, ближайших к его центру: так делает и ACE, когда показывает найденные концепты.

In [ ]:
def show_clusters(clusters, cluster_of, dist, titles, n=6):
    fig, axs = plt.subplots(len(clusters), n, figsize=(2 * n, 2.1 * len(clusters)))
    for row, c in enumerate(clusters):
        members = np.where(cluster_of == c)[0]
        nearest = members[np.argsort(dist[members, c])[:n]]
        for j in range(n):
            axs[row, j].axis('off')
            if j < len(nearest):
                axs[row, j].imshow(patches[nearest[j]])
        axs[row, 0].set_title(titles[row], loc='left', fontsize=11)
    plt.tight_layout()
    plt.show()

show_clusters(kept, cluster_of, dist, [f'cluster {c}: {(cluster_of == c).sum()} segments' for c in kept])

## Шаг 4. TCAV-score каждого кластера

Теперь каждый оставшийся кластер — это концепт в смысле TCAV: набор картинок-примеров. Дальше все как в уроке «TCAV: Практика»: для пары «кластер против случайных изображений» `captum` обучает линейный классификатор на активациях `inception4e`, берет его нормаль как вектор концепта (CAV) и считает, у какой доли зебр производная логита класса 340 по направлению CAV положительна. Это и есть TCAV-score (`sign_count`).

Из каждого кластера берем 40 сегментов, ближайших к центру, чтобы число примеров концепта было близко к числу случайных изображений (38 и 41 в `random500_0` и `random500_1`). Каждый кластер сравниваем с двумя случайными наборами и усредняем два TCAV-score. В статье случайных наборов десятки, и важным считают концепт, чей TCAV-score статистически отличается от TCAV-score случайного концепта; с двумя наборами такой проверки нет, поэтому TCAV-score около 0.5 здесь не отличить от шума.

In [ ]:
N_CONCEPT = 40

def make_concept(cid, name, imgs):
    tensors = [normalize(img).to(device) for img in imgs]
    return Concept(id=cid, name=name, data_iter=torch.utils.data.DataLoader(tensors, batch_size=64))

random_0_imgs, random_1_imgs = load_images('concepts/random500_0'), load_images('concepts/random500_1')
print(f'random500_0: {len(random_0_imgs)} images, random500_1: {len(random_1_imgs)} images')
random_0_concept = make_concept(100, 'random500_0', random_0_imgs)
random_1_concept = make_concept(101, 'random500_1', random_1_imgs)

def cluster_concept(c, cluster_of, dist):
    members = np.where(cluster_of == c)[0]
    nearest = members[np.argsort(dist[members, c])[:N_CONCEPT]]
    return make_concept(c, f'cluster_{c}', [patches[j] for j in nearest])

def cluster_tcav(clusters, cluster_of, dist, save_path):
    shutil.rmtree(save_path, ignore_errors=True)   # captum кэширует CAV на диске по номеру концепта
    tcav = TCAV(model=model, layers=[LAYER], save_path=save_path)   # чувствительность — градиент по активациям слоя, умолчание Captum
    concepts = [cluster_concept(c, cluster_of, dist) for c in clusters]
    experimental_sets = [[cc, r] for cc in concepts for r in (random_0_concept, random_1_concept)]

    # Сиды до обучения классификаторов CAV: captum перемешивает примеры через random и обучает классификатор в torch
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    scores = tcav.interpret(inputs=zebra_tensors, experimental_sets=experimental_sets, target=zebra_ind)
    return {c: np.mean([scores[f'{c}-{r}'][LAYER]['sign_count'][0].item() for r in (100, 101)]) for c in clusters}

In [ ]:
tcav_scores = cluster_tcav(kept, cluster_of, dist, './cav_k12/')
ranking = sorted(kept, key=lambda c: -tcav_scores[c])
for place, c in enumerate(ranking, 1):
    print(f'{place:2d}. cluster {c:2d}: TCAV score = {tcav_scores[c]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar([f'cluster {c}' for c in ranking], [tcav_scores[c] for c in ranking], color='mediumspringgreen')
ax.bar_label(bars, fmt='%.3f')
ax.axhline(0.5, color='gray', ls='--', lw=1)
ax.set_ylim(0, 1.1)
ax.set_ylabel('TCAV score (sign_count), mean of 2 random sets')
ax.set_title(f'ACE concepts for class "zebra", layer {LAYER}, k={K}')
plt.xticks(rotation=45)
plt.show()

Столбики упорядочены по TCAV-score от большего к меньшему; штриховая линия — 0.5, уровень, который дал бы концепт без влияния на класс. Посмотрим на сегменты кластеров в том же порядке.

In [ ]:
show_clusters(ranking, cluster_of, dist, [f'#{p}: cluster {c}, TCAV score {tcav_scores[c]:.3f}' for p, c in enumerate(ranking, 1)])

**Задание 4 (вопрос для самопроверки).** Какой кластер получил самый высокий TCAV-score, чему он равен и как выглядят сегменты этого кластера? Какое имя вы бы дали этому концепту?

<details><summary>Проверить себя</summary>

Самый высокий TCAV-score, 0.375, у кластера 6: его сегменты — полосы головы и шеи зебры крупным планом, и подходящее имя для него — «полосы зебры». Следом идут кластер 10 (0.225) с тонкими полосами на ногах и по краям тела и кластер 5 (0.175) с крупными кусками тела в полосах. Кластер 5 — единственный, где больше половины сегментов взято с разрешения 15 (59 из 78).

До 0.5 не дошел ни один кластер: на слое `inception4e` производная логита класса «зебра» по направлению каждого найденного концепта положительна меньше чем у половины зебр. Так же на этом слое ведет себя и концепт `striped` из урока «TCAV: Практика»: 0.34 и 0.17 против двух случайных наборов. С двумя случайными наборами такие значения от шума не отличить.

</details>


## Ограничение: кластер — еще не концепт

Имя концепту дали мы, глядя на 6 картинок, а сам кластер — результат KMeans с выбранным нами $k$. Проверим, насколько найденные концепты зависят от этого выбора: повторим шаги 2–4 на тех же активациях с $k=6$ и сравним.

Для каждого концепта, найденного при $k=12$, найдем самый похожий кластер из найденных при $k=6$ по коэффициенту Жаккара: число сегментов, попавших в оба кластера, деленное на число сегментов, попавших хотя бы в один. Коэффициент 1 значит, что кластер воспроизвелся целиком, значение около 0 — что такого концепта при $k=6$ нет.

In [ ]:
K_ALT = 6
kmeans_alt = KMeans(n_clusters=K_ALT, n_init=10, random_state=0).fit(feats)
cluster_alt, dist_alt = kmeans_alt.labels_, kmeans_alt.transform(feats)
kept_alt = select_clusters(cluster_alt, owners)
tcav_alt = cluster_tcav(kept_alt, cluster_alt, dist_alt, './cav_k6/')
ranking_alt = sorted(kept_alt, key=lambda c: -tcav_alt[c])

print(f'k={K_ALT}: kept {len(kept_alt)} of {K_ALT} clusters')
for place, c in enumerate(ranking_alt, 1):
    print(f'{place:2d}. cluster {c:2d}: {(cluster_alt == c).sum():4d} segments, TCAV score = {tcav_alt[c]:.3f}')

print(f'\nk={K} concept -> most similar k={K_ALT} cluster')
for place, c in enumerate(ranking, 1):
    a = cluster_of == c
    jac = [(a & (cluster_alt == d)).sum() / (a | (cluster_alt == d)).sum() for d in range(K_ALT)]
    d = int(np.argmax(jac))
    alt = f'TCAV score {tcav_alt[d]:.3f}' if d in kept_alt else 'dropped by the filter'
    print(f'#{place:2d} cluster {c:2d} (TCAV score {tcav_scores[c]:.3f}) -> cluster {d} (Jaccard {jac[d]:.2f}, {alt})')

In [ ]:
show_clusters(ranking_alt, cluster_alt, dist_alt, [f'k={K_ALT}, #{p}: cluster {c}, TCAV score {tcav_alt[c]:.3f}' for p, c in enumerate(ranking_alt, 1)])

**Что показал эксперимент с $k=6$.** При $k=6$ фильтр прошли 5 кластеров из 6, и TCAV-score 0.5 не набрал ни один из них: кластер 4 (237 сегментов) получил 0.275, кластер 1 (164 сегмента) 0.175, кластер 3 (165 сегментов) 0.100, кластер 2 (124 сегмента) 0.050, кластер 0 (191 сегмент) 0.025. Верхние концепты, найденные при $k=12$, разошлись по ним по-новому: кластеры 10 и 5 (тонкие полосы и крупные куски с полосами) слились в кластер 1 с коэффициентами Жаккара 0.32 и 0.48; кластер 6 (полосы головы и шеи) больше всего похож на кластер 2, но с коэффициентом всего 0.18; кластер 2 (полосатые куски вперемешку с посторонними) ушел в кластер 4 вместе с кластером 0 (темные куски со светлой полосой). Ни один концепт из найденных при $k=12$ не воспроизвелся при $k=6$ с коэффициентом Жаккара выше 0.49.

Сколько концептов про полосы найдено, где проходят границы между ними и какой TCAV-score у каждого, задает выбранное нами $k$: кластер 6, у которого при $k=12$ самый высокий TCAV-score 0.375, больше всего похож на кластер с TCAV-score 0.050 при $k=6$. Кластер — это результат KMeans, а концептом его делает человек, который на него смотрит.


## Заключение

В этой практике вы прошли три шага ACE на 20 зебрах:
- SLIC на разрешениях 15 и 50 дал 982 сегмента, после отбрасывания повторов осталось 954; каждый сегмент подан сети на сером фоне 117;
- KMeans по усредненным активациям слоя `inception4e` разбил их на 12 кластеров, фильтр по числу сегментов и числу изображений оставил 10;
- TCAV-score против двух случайных наборов ни у одного кластера не дошел до 0.5. Выше остальных — три кластера с полосами зебры: полосы головы и шеи (кластер 6, 0.375), тонкие полосы (кластер 10, 0.225) и крупные куски тела (кластер 5, 0.175); темные куски со светлой полосой (кластер 0) и полосатые куски вперемешку с посторонними (кластер 2) получили 0.125 и 0.100, остальные пять кластеров (зеленая и сухая трава, размытый фон, однотонные светлые пятна) — не выше 0.025.

Все три ограничения из теории видны на этих данных:
- **кластер — еще не концепт:** при $k=6$ полосатые концепты перегруппировались, кластер 6 больше всего похож на кластер с TCAV-score 0.050, и имя каждому кластеру дали мы, глядя на 6 картинок;
- **нарезка решает за метод:** крупные куски тела с полосами (кластер 5) собраны в основном из сегментов разрешения 15 (59 из 78); без крупного разрешения такого концепта бы не было;
- **обрывки на сером фоне — вне распределения:** сегменты кластера 6 с самым высоким TCAV-score 0.375 модель сама относит к классу «зебра» лишь в 9% случаев, хотя целые изображения — в 20 случаях из 20; высокая доля только у кластера 5 с крупными кусками тела (73%), а TCAV-score у него 0.175. TCAV измеряет чувствительность логита к направлению в пространстве активаций, а не то, узнает ли модель зебру в обрывке.

И еще одно упрощение: в статье TCAV-score концепта сравнивают с TCAV-score случайных концептов статистическим тестом по десяткам случайных наборов, а у нас наборов два, поэтому значения вроде 0.375 и 0.225 у кластеров 6 и 10 от шума не отличить.

Оригинальная реализация ACE на TensorFlow — [github.com/amiratag/ACE](https://github.com/amiratag/ACE).

**Спасибо за работу и внимание!**